In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU is available.")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}") # Assuming at least one GPU
else:
    print("GPU is not available.")

GPU is available.
Number of GPUs: 1
Current GPU: 0
GPU name: Tesla T4


In [ ]:
import sys
!{sys.executable} -m pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 34.6 MB/s eta 0:00:00


## Helper functions

* `load_and_process_video()` preprocesses videos to be later manipulated. All videos to be manipulated should share the same parameters: `height`, `width`, and `max_frames`
* `get_latents_chunked()` produces a latent tensor representing the video passed as a paramter according to the `AutoencoderKLCogVideoX` VAE. The `chunk_size` parameter can be manipulated to vary the amount of VRAM the function will require
* `decode_latents_chunked()` produces a video based on the latent video representation passed to it. This function also contains the `chunk_size` variable to control VRAM usage

In [ ]:
from diffusers import AutoencoderKLCogVideoX
from diffusers.utils import export_to_video
import gc
import numpy as np

import torchvision
from torchvision import transforms

torch.cuda.empty_cache()
gc.collect()

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load 3d vae
# use float16 to save memory
model_id = "THUDM/CogVideoX-2b"
vae = AutoencoderKLCogVideoX.from_pretrained(
    model_id,
    subfolder='vae',
    torch_dtype=torch.float16,
).to(device)

# enables tiling which should save on vram usage
vae.enable_tiling()

# helper to prep video for vae
def load_and_process_video(video_path, height=480, width=720, max_frames=16):
    print(f"Loading {video_path}...")

    # read video
    video_frames, _, _ = torchvision.io.read_video(video_path, output_format="TCHW", pts_unit='sec')

    # cap frames
    if len(video_frames) > max_frames:
        video_frames = video_frames[:max_frames]

    current_frames = len(video_frames)

    # transform pipeline
    transform = transforms.Compose([
        transforms.Resize(height, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop((height, width)),
    ])

    # apply transforms
    processed_frames = torch.stack([transform(f) for f in video_frames])

    # format for VAE
    video_tensor = processed_frames.permute(1, 0, 2, 3).unsqueeze(0)

    # normalize
    video_tensor = video_tensor.float() / 255.0  # Now [0, 1]
    video_tensor = (video_tensor * 2.0) - 1.0    # Now [-1, 1]

    return video_tensor.to(device, dtype=torch.float16)

# manually splits vid into temporal chunks to save vram
def get_latents_chunked(video_tensor, chunk_size=4):
    frames = video_tensor.shape[2]
    latent_list = []

    with torch.no_grad():
        for i in range(0, frames, chunk_size):
            # get slice of frame
            end = min(i + chunk_size, frames)
            video_chunk = video_tensor[:,:,i:end,:,:]

            # encode current chunk
            posterior = vae.encode(video_chunk).latent_dist
            latents = posterior.sample() * vae.config.scaling_factor
            latent_list.append(latents)

            # clean up vram
            del video_chunk, posterior, latents
            torch.cuda.empty_cache()

    # stitch chunks together and return
    return torch.cat(latent_list, dim=2)


# decode latents & manually split to save vram
def decode_latents_chunked(latents, chunk_size=1):
    latent_frames_count = latents.shape[2]
    decoded_video_list = []

    with torch.no_grad():
        for i in range(0, latent_frames_count, chunk_size):
            # slice latent
            end = min(i+chunk_size, latent_frames_count)
            latent_chunk = latents[:,:,i:end,:,:]

            # decode current chunk
            frames = vae.decode(latent_chunk).sample

            # move to cpu to free vram
            frames = (frames / 2 + 0.5).clamp(0,1)
            decoded_video_list.append(frames.cpu())

            # cleanup
            del latent_chunk, frames
            torch.cuda.empty_cache()
    # stitch together and return
    return torch.cat(decoded_video_list, dim=2)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/862M [00:00<?, ?B/s]

## Loading videos

In [ ]:
video_a = load_and_process_video('/content/King_Walking_Video_Generated.mp4', max_frames=64)
video_b = load_and_process_video('/content/Queen_Walking_Video_Generated.mp4', max_frames=64)

Loading /content/King_Walking_Video_Generated.mp4...
Loading /content/Queen_Walking_Video_Generated.mp4...


## Generate latent representaitons

In [ ]:
latents_a = get_latents_chunked(video_a)
latents_b = get_latents_chunked(video_b)

## Manipulate latent representations

In [ ]:
# manipulation
# alpha = 0.5
# latents_hybrid = (latents_a * (1-alpha)) + (latents_b * alpha)
latents_hybrid = latents_a - latents_b

## Decode and save video

In [ ]:
print('decoding hybrid vid')
decoded_frames = decode_latents_chunked(latents_hybrid)

video_tensor = decoded_frames[0]
video_tensor = video_tensor.permute(1,2,3,0)
video_np = video_tensor.cpu().numpy()
video_np = (video_np * 255).astype(np.uint8)

print('saving vid')
export_to_video(video_np, 'output.mp4', fps=16)

decoding hybrid vid
saving vid


'output.mp4'

## Display Video

In [ ]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open('output.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

# Pull in Kaggle Dataset
The goal here is to be able to pull in a video dataset that has labeled videos so we can attempt to generate average latent representations of the videos. These (hopeful) representations of the classes from the dataset can be added to move the latent representation of our input video so it depicts a different video.

This may not work for a few reasons.
1. Moving in the direction of a different idea does not imply the latent representation will produce a meaningful output video as-is. This is a similar idea to how adding two words together might have a meaning in the latent/embedding space, but we use cosine similarity to find the nearest token/word to illustrate a meaningful output.
2. We might simply not have enough data from any given class to produce the average latent representation we hope for.

Note: our ideal dataset was too large to pull into a colab runtime, so it was downloaded to a local machine and loaded in manually.

Dataset URL: https://www.kaggle.com/datasets/rohanmallick/kinetics-train-5per

Instead we manually upload videos from the following classes as representatives:

* archery
* bartending
* cartwheeling
* dodgeball
* headbanging
* jogging
* laughing
* motorcycling
* sailing
* texting
* welding
* yoga
* zumba

In [ ]:
import zipfile
import os

zip_path = "/content/vids.zip"

print(f"Unzipping {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("videos")

print("Done! Files are ready.")

Unzipping /content/vids.zip...
Done! Files are ready.


## Latent Video Averaging

In [ ]:
from pathlib import Path
import torch

ROOT_DIR = '/content/videos/project_vids'
CLASSES = [d for d in os.listdir(ROOT_DIR) if os.path.isdir(os.path.join(ROOT_DIR, d))]
VALID_EXTENSIONS = ['.mp4']
OUTPUT_FILE = 'class_average_latents.pt'

def is_valid_file(fpath):
  return Path(fpath).suffix.lower() in VALID_EXTENSIONS

def process_dir(classification: str):
  dir_path = os.path.join(ROOT_DIR, classification)
  files = [f for f in os.listdir(dir_path) if is_valid_file(os.path.join(dir_path, f))]
  file_count = len(files)
  latents = []

  for file in files:
    processed_video = load_and_process_video(os.path.join(dir_path, file))
    latent_video = get_latents_chunked(processed_video)
    latents.append(latent_video)

  sum_tensor = torch.zeros_like(latents[0])
  for tensor in latents:
    sum_tensor += tensor

  avg_tensor = sum_tensor / file_count
  return avg_tensor


# main
class_tensor_dict = {}

for classification in CLASSES:
  print(f'Processing class: {classification}...')
  class_average = process_dir(classification)
  class_tensor_dict[classification] = class_average

torch.save(class_tensor_dict, OUTPUT_FILE)



Processing class: yoga...
Loading /content/videos/project_vids/yoga/vrBEJhM3n-U.mp4...


/usr/local/lib/python3.12/dist-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


Loading /content/videos/project_vids/yoga/1hq8VgJFDnI.mp4...
Loading /content/videos/project_vids/yoga/xJwAYKUx7uA.mp4...
Loading /content/videos/project_vids/yoga/jyppenSMnZs.mp4...
Loading /content/videos/project_vids/yoga/6hV0sF52cWQ.mp4...
Loading /content/videos/project_vids/yoga/sROozDIfa_A.mp4...
Loading /content/videos/project_vids/yoga/0X79m_8GlOE.mp4...
Loading /content/videos/project_vids/yoga/vqkw-ALhCsA.mp4...
Loading /content/videos/project_vids/yoga/cGkE_XlPDq0.mp4...
Loading /content/videos/project_vids/yoga/BY_mTEDi7V8.mp4...
Loading /content/videos/project_vids/yoga/RSMcZGW93GA.mp4...
Loading /content/videos/project_vids/yoga/sskng5LcHEY.mp4...
Loading /content/videos/project_vids/yoga/4Jy34UQZdD8.mp4...
Loading /content/videos/project_vids/yoga/rTqITqnOz4w.mp4...
Loading /content/videos/project_vids/yoga/2VGFcJ84qeo.mp4...
Loading /content/videos/project_vids/yoga/uu-JzxgtERE.mp4...
Loading /content/videos/project_vids/yoga/rGp9LsA39RU.mp4...
Loading /content/videos/